# EDA of `r/litigi`

In [1]:
from functions.utilities import preprocess
import pandas as pd
import plotly.express as px

## Load and preprocess the data

In [2]:
filename='processed_data\itigi_comments_copnsolidated.parqet'
litigi=pd.read_parquet(filename).drop_duplicates(subset=['author','body','id', 'created_utc'])
litigi['created_dt']=(pd.to_datetime(litigi['created_utc'],unit='s'))
litigi['body_preprocessed']=litigi['body'].apply(lambda x: preprocess(x))
id_to_author = litigi.set_index('id')['author'].to_dict()
litigi['parent_author'] = litigi['parent_id'].str.split('_').str[1].map(id_to_author)
litigi.head()

<>:1: SyntaxWarning: invalid escape sequence '\i'
<>:1: SyntaxWarning: invalid escape sequence '\i'
C:\Users\Alessandro\AppData\Local\Temp\ipykernel_14992\3846726544.py:1: SyntaxWarning: invalid escape sequence '\i'
  filename='processed_data\itigi_comments_copnsolidated.parqet'


,author,author_flair_css_class,author_flair_text,body,can_gild,collapsed,collapsed_reason,controversiality,created_utc,distinguished,...,top_awarded_type,comment_type,collapsed_reason_code,retrieved_utc,author_is_blocked,unrepliable_reason,editable,created_dt,body_preprocessed,parent_author
0,Trickyhere,None,None,Qualcosa da spaccare?,True,False,None,0,1502873647,None,...,None,None,None,NaN,None,NaN,None,2017-08-16 08:54:07,Qualcosa da spaccare?,NaN
1,panicClark,None,None,"Avevo quest'ideona dello sticky giornaliero ""A...",True,False,None,0,1502873820,None,...,None,None,None,NaN,None,NaN,None,2017-08-16 08:57:00,"Avevo quest'ideona dello sticky giornaliero ""A...",NaN
2,MeccAnon,None,None,"Risposta USENET: ""non l'avete capita"" (cit.)\n...",True,False,None,0,1502874353,None,...,None,None,None,NaN,None,NaN,None,2017-08-16 09:05:53,"Risposta USENET: ""non l'avete capita"" (cit.) ...",panicClark
3,panicClark,None,None,[GENIO.](https://www.reddit.com/r/litigi/comme...,True,False,None,0,1502876399,moderator,...,None,None,None,NaN,None,NaN,None,2017-08-16 09:39:59,GENIO.,MeccAnon
4,MeccAnon,None,None,&gt; nella sidebar trovate il suo manifesto or...,True,False,None,0,1502877513,None,...,None,None,None,NaN,None,NaN,None,2017-08-16 09:58:33,&gt; nella sidebar trovate il suo manifesto or...,NaN


In [3]:
litigi.describe()

,controversiality,created_utc,gilded,score,removal_reason,author_created_utc,total_awards_received,retrieved_utc,unrepliable_reason,created_dt
count,309848.000000,3.098480e+05,309848.000000,309848.000000,0.0,2.068800e+05,278361.000000,5.037900e+04,0.0,309848
mean,0.006648,1.614021e+09,0.000045,3.039235,NaN,1.518252e+09,0.003553,1.627637e+09,NaN,2021-02-22 19:05:35.494362112
min,0.000000,1.502874e+09,0.000000,-18.000000,NaN,1.161160e+09,0.000000,1.626817e+09,NaN,2017-08-16 08:54:07
25%,0.000000,1.602019e+09,0.000000,1.000000,NaN,1.469449e+09,0.000000,1.627189e+09,NaN,2020-10-06 21:15:41.750000128
50%,0.000000,1.626860e+09,0.000000,2.000000,NaN,1.538932e+09,0.000000,1.627734e+09,NaN,2021-07-21 09:32:46.500000
75%,0.000000,1.638623e+09,0.000000,4.000000,NaN,1.591986e+09,0.000000,1.627989e+09,NaN,2021-12-04 13:08:06
max,1.000000,1.672531e+09,1.000000,38.000000,NaN,1.671312e+09,5.000000,1.628436e+09,NaN,2022-12-31 23:51:49
std,0.081266,3.681197e+07,0.006722,2.514462,NaN,9.638735e+07,0.067956,4.731565e+05,NaN,NaN


In [4]:
monthly_contrib = litigi.groupby(litigi['created_dt'].dt.to_period("M")).agg(
    Count=('controversiality', 'size')
).reset_index()

# Convert period to timestamp for proper plotting
monthly_contrib.rename(columns={'created_dt': 'Month'}, inplace=True)
monthly_contrib['Month'] = monthly_contrib['Month'].dt.to_timestamp()

# Create bar plot
fig = px.bar(monthly_contrib, x='Month', y='Count',
             labels={'Month': 'Month', 'Count': 'Number of Posts'},
             title="Monthly User Comments on r/litigi",
             color_discrete_sequence=["#1f77b4"],color='Count',color_continuous_scale='viridis')  # High-contrast blue

# Improve layout
fig.update_layout(
    xaxis=dict(
        tickformat="%b %Y",  # Show abbreviated month and year
        tickmode="auto",      # Auto-adjusts to fit the range
        showgrid=True
    ),
    yaxis=dict(showgrid=True),
    title_x=0.5  # Centers the title
)

fig.show()

In [5]:
hour_activity = litigi.groupby(litigi['created_dt'].dt.hour).agg(
    Count=('controversiality', 'size')
).reset_index()

# Rename column for clarity
hour_activity.rename(columns={'created_dt': 'Hour'}, inplace=True)
fig = px.bar(hour_activity, x='Hour', y='Count', 
             labels={'Hour': 'Hour of the Day', 'Count': 'Number of Posts'},
             title="Hourly Activity on r/litigi",
             color='Count',color_continuous_scale='viridis')
fig.update_layout(
    xaxis=dict(
        tickmode='linear',
        tickvals=list(range(24)),
    ),
    yaxis=dict(showgrid=True),
    title_x=0.5  # Centers title
)

fig.show()


In [6]:
week_activity = litigi.groupby(litigi['created_dt'].dt.day_of_week).agg(
    Count=('controversiality', 'size')
).reset_index()
days=['Lun','Mar','Mer','Gio','Ven','Sab','Dom']
week_activity['Weekday']=days
# Rename column for clarity
#week_activity.rename(columns={'created_dt': 'Weekday'}, inplace=True)
fig = px.bar(week_activity, x='Weekday', y='Count', 
             labels={'Weekday': 'Day of the week', 'Count': 'Number of Posts'},
             title="Weekly Activity on r/litigi",
             color='Count',color_continuous_scale='viridis')
fig.update_layout(
    xaxis=dict(
        tickmode='linear',
        tickvals=days
    ),
    yaxis=dict(showgrid=True),
    title_x=0.5  # Centers title
)

fig.show()

## Evolution of hourly posting

In [7]:
import pandas as pd
import plotly.express as px

# Ensure 'created_dt' is a datetime column

# Extract relevant time features
litigi['year_month'] = litigi['created_dt'].dt.to_period('M')  # Year-Month format
litigi['hour'] = litigi['created_dt'].dt.hour  # Hour of the day

# Count occurrences for each (year_month, hour) pair
heatmap_data = litigi.groupby(['year_month', 'hour']).size().reset_index(name='count')

# Convert 'year_month' to string for Plotly
heatmap_data['year_month'] = heatmap_data['year_month'].astype(str)

# Create the heatmap
fig = px.imshow(
    heatmap_data.pivot(columns='year_month', index='hour', values='count'),
    labels=dict(x="Hour of Day", y="Month", color="Post Count"),
    y=list(range(24)),  # Ensure x-axis is in correct order
    x=heatmap_data['year_month'].unique(),
    color_continuous_scale="Viridis"
)

fig.update_layout(
    title="User Posting Habits Evolution (Hourly Distribution per Month)",
    xaxis_title="Month",
    yaxis_title="Hour of Day",
    yaxis=dict(autorange="reversed"),  # To have the latest month on top
    coloraxis_colorbar=dict(title="Post Count")
)

fig.show()


In [12]:
# Convert timestamp
litigi['created_dt'] = pd.to_datetime(litigi['created_utc'], unit='s')

# Extract features
litigi['month'] = litigi['created_dt'].dt.to_period('M').astype(str)
litigi['hour'] = litigi['created_dt'].dt.hour

# Aggregate and normalize by month
post_counts = pd.crosstab(litigi['month'], litigi['hour'])
post_counts = post_counts.reindex(columns=range(24), fill_value=0)
post_counts = post_counts.sort_index()

# Normalize to get hourly percentages within each month
df_normalized = post_counts.div(post_counts.sum(axis=1), axis=0) * 100

# Transpose to get months on x-axis and hours on y-axis
df_normalized = df_normalized.T

# Create heatmap
fig = px.imshow(
    df_normalized,
    labels={'x': 'Month', 'y': 'Hour of Day', 'color': 'Post Frequency (%)'},
    x=df_normalized.columns,
    y=df_normalized.index,
    color_continuous_scale='Viridis',
    title='Normalized Hourly Posting Patterns by Month',
    aspect='auto'
)

# Customize layout
fig.update_layout(
    xaxis_title='Month',
    yaxis_title='Hour of Day (UTC)',
    xaxis=dict(tickangle=45, tickmode='array', tickvals=df_normalized.columns[::2]),
    yaxis=dict(tickvals=list(range(24))),
    coloraxis_colorbar=dict(title='% of Posts'),
    height=600  # Taller plot for better hour labels
)

# Add hover template
fig.update_traces(
    hovertemplate='<b>Month</b>: %{x}<br><b>Hour</b>: %{y}:00 UTC<br><b>Frequency</b>: %{z:.2f}%<extra></extra>'
)

fig.show()